# v11 D (GPU 1): learning a NEW language (Indonesian) after a count-matched lesion / under silencing, 5 seeds
- Starts: distant phase 20 and related phase 16 (both before any breakdown) + a fresh untrained model as the baseline.
- Training: 1,500 steps on English -> Indonesian, learning rate 1e-4. Score: chrF on 200 FLORES sentences (step 0 and end).
- Conditions: untouched · self@1.0 vs task@1.0 · self@0.5 vs task@0.5 vs look-alike@0.5, for **lesion** and for **silence**. Seed 0 of everything first.
Results: `analysis/v11/train_indonesian/` · progress: `tail -f logs/v11_D_indonesian.log`

In [ ]:
import os, sys, glob, json, time, subprocess
os.environ["HF_HUB_DISABLE_XET"] = "1"
GPU = "1"; LOG = "v11_D_indonesian"
REPO = "/workspace/shreshth"; os.chdir(REPO); sys.path.insert(0, REPO)
from IPython.display import Image, display
from huggingface_hub import HfApi, hf_hub_download
FILES, HF_REPO = [], None
for attempt in range(5):
    try:
        api = HfApi(); HF_REPO = api.whoami()["name"] + "/cml-lang-self-checkpoints"; FILES = api.list_repo_files(HF_REPO); break
    except Exception as e:
        print("HF retry", attempt, type(e).__name__); time.sleep(10)
if not FILES:
    print("could not list the HF repo: only checkpoints already on this machine will be used")

def have(run, ph):
    return bool(glob.glob(f"runs/{run}/ckpt/p{ph:03d}_*.pt"))

def get(run, phases):
    """download what is missing; returns the phases that are on this machine afterwards"""
    ok = []
    for ph in phases:
        if not have(run, ph):
            name = [f for f in FILES if f.startswith(f"{run}/ckpt/p{ph:03d}_")]
            if name:
                try:
                    hf_hub_download(HF_REPO, name[0], local_dir="runs")
                except Exception as e:
                    print("download failed:", run, ph, type(e).__name__, e)
        if have(run, ph):
            ok.append(ph)
        else:
            print("not available (not here, not on HF):", run, ph)
    return ok

def run(langset, *cmd):
    """one call of core_v11.py; output shown here AND appended to logs/<LOG>.log (watch with: tail -f logs/<LOG>.log)"""
    env = {**os.environ, "CML_LANGSET": langset}; os.makedirs("logs", exist_ok=True)
    p = subprocess.Popen([sys.executable, "-u", "core_v11.py", "--gpu", GPU, *cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    with open(f"logs/{LOG}.log", "a") as log:
        for line in p.stdout:
            print(line, end=""); log.write(line); log.flush()
    p.wait(); print("exit code", p.returncode)

def summary(part, out=None):
    cmd = [sys.executable, "core_v11.py", "--summarize", "--part", part] + (["--out", out] if out else [])
    r = subprocess.run(cmd, capture_output=True, text=True); print(r.stdout[-12000:]); print(r.stderr[-3000:])

DIST, HIN, MIX, REL = "v4b_continual_s0", "v4b_only_hi_s0", "v6b_mixed_s0", "v8_indic_continual_s0"

In [ ]:
r = subprocess.run([sys.executable, "-u", "prep_newtasks.py"], capture_output=True, text=True); print(r.stdout[-3000:]); print(r.stderr[-3000:])
print("Indonesian data there:", all(os.path.exists(f"data/newtask/indonesian/{s}.jsonl") for s in ["train", "val", "test"]))

In [ ]:
STARTS = [("distant", "distant", DIST, 20), ("indic", "related", REL, 16)]
COMMON = ["--part", "train", "--task", "indonesian", "--lr", "1e-4", "--steps", "1500"]
for seeds in ["0", "1,2,3,4"]:
    for langset, fam, r, ph in STARTS:
        if get(r, [ph]):
            for mode in ["lesion", "silence"]:
                run(langset, *COMMON, "--mode", mode, "--family", fam, "--model", f"{fam}{ph}={r}:{ph}", "--seeds", seeds)
    run("distant", *COMMON, "--family", "fresh", "--model", "fresh", "--seeds", seeds)
    summary("train", "train_indonesian")

In [ ]:
summary("train", "train_indonesian")
if os.path.exists("analysis/v11/train_indonesian/curves.png"): display(Image("analysis/v11/train_indonesian/curves.png"))